In [0]:
from common_utils.ingester import read_raw, write_table
from common_utils.logging import get_logger
import json

# =========================================
# 0. Logger
# =========================================

logger = get_logger("Bronze_ingestion")
logger.info("Session started")

# =========================================
# 1. Read configuration
# =========================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")

with open(config_path, "r") as f:
    config = json.load(f)

# =========================================
# 2. Configuration sections
# =========================================

source_config = config["source"]
target_config = config["target"]
sources = config["sources"]

# =========================================
# 3. Prepare paths
# =========================================

run_date = source_config["load_date"]

source_folder = (
    f"{source_config['base_path']}/"
    f"load_date={run_date}"
)

target_schema = (
    f"{target_config['catalog']}."
    f"{target_config['schema']}"
)

# =========================================
# 4. Ingestion
# =========================================

for file_name, table_name in sources.items():

    source_path = f"{source_folder}/{file_name}"

    target_table = f"{target_schema}.{table_name}"

    logger.info(f"Reading source file: {source_path}")

    df = read_raw(spark, source_path)

    logger.info(f"Writing into Bronze table: {target_table}")

    write_table(df, target_table)

    logger.info(f"Successfully ingested: {target_table}")

# =========================================
# 5. Completion
# =========================================

logger.info(
    f"Successfully ingested all {len(sources)} files "
    f"for {run_date}"
)